# Chapter 16: Supply-Chain Security: Pinning, Injection, Attestations (notebook edition)

## Learning Objectives

- Classify and verify pins
- Reproduce injection and fix it
- Compare PR triggers and read policy rejections

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

GHA_MODE = os.environ.get("GHA_MODE", "fixture")  # "fixture" | "live"
GHA_REPO = os.environ.get("GHA_REPO", "Friend09/practice_git_intro_to_github_actions")
print("mode:", GHA_MODE, "| repo:", GHA_REPO)

## 1. Pin classification

Only a full 40-hex SHA is immutable; a short SHA or a tag moves.

In [ ]:
from intro_gha.pinning import classify
SHA = "3d3c42e5aac5ba805825da76410c181273ba90b1"
for ref in [f"actions/checkout@{SHA}", "actions/checkout@v7", "actions/checkout@" + SHA[:7], "./local", "docker://alpine:3.20"]:
    print(f"{ref[:44]:46} -> {classify(ref)}")
assert classify(f"actions/checkout@{SHA}") == "sha" and classify("actions/checkout@" + SHA[:7]) == "tag-or-branch"

## 2. The wrong kind of SHA

The annotated tag object and the commit are different SHAs for the same version; Dependabot proposed the swap.

In [ ]:
import json
s = json.loads((ROOT/"fixtures"/"security_ch16_summary.json").read_text())
tv = s["pinning"]["tag_object_vs_commit"]
print(tv["ref_object_type"], tv["tag_object_sha"][:8], "->", tv["commit_sha"][:8])
assert tv["ref_object_type"] == "tag" and tv["tag_object_sha"] != tv["commit_sha"]

## 3. Script injection

GitHub pastes the expression before the shell runs, so the payload becomes code. We reproduce the substitution and compare it with the real command line.

In [ ]:
from intro_gha.injection import interpolate, untrusted_expressions
inj = s["injection"]
script = 'echo "REPORT vulnerable_title=${{ inputs.title }}"'
rendered = interpolate(script, {"inputs.title": inj["payload"]})
print(rendered)
assert rendered == inj["vulnerable"]["command_line_run"]
print("flagged:", untrusted_expressions(script), "| safe form flagged:", untrusted_expressions('echo "$TITLE"'))

## 4. pull_request vs pull_request_target

The same PR: different workflow file, different checkout, different SHA.

In [ ]:
pr = s["pr_context"]
for ev in ("pull_request", "pull_request_target"):
    r = pr[ev]; print(f"{ev:20} workflow={r['workflow_version']:13} marker={r['marker']:13} sha={r['github_sha']} ref={r['ref']}")
print("head checked out under target ->", pr["pull_request_target"]["head_marker_under_target"])
assert pr["pull_request_target"]["head_marker_under_target"] == "head-content"

## 5. Policies and attestation

Pinning policy fails the unpinned job at `Set up job`; the allow list kills the whole run; a tampered wheel has no attestation.

In [ ]:
print(s["sha_policy"]["on"]); print(s["allow_list"]["restricted"]["conclusion"], s["allow_list"]["restricted"]["jobs"], "jobs")
a = s["attestation"]; print(a["verify_in_workflow"], "|", a["tampered"][:48])
assert s["allow_list"]["restricted"]["conclusion"] == "startup_failure" and a["wheel_digest"] != a["tampered_digest"]

## 6. Rulesets and required checks

A required check is matched by **name**. We read what the four demo pull requests showed and confirm `CODEOWNERS` owns the workflow directory.

In [ ]:
rs = json.loads((ROOT / "fixtures" / "rulesets_ch16.json").read_text())
for n, pr in rs["prs"].items():
    print(f"PR #{n}: {pr['case']:55} ci-ok={pr['ci_ok']} -> {pr.get('mergeable_state') or pr['mergeStateStatus']}")
print("renamed check:", rs["rename_variant"])
print("tag rules:", rs["tag_ruleset"]["rules"])
print((ROOT / ".github" / "CODEOWNERS").read_text())

## Takeaways & Next Steps

- Pin commits, verify pins are commits, enforce with policy.
- Values go through `env:`, never inline.
- Never check out PR head under `pull_request_target`.
- Next: Chapter 17, monitoring and debugging.

## Chapter Link

Read: `learning_modules/chapter_16_supply_chain_security.md`